# Complete Periodicity Analysis with Visual Feedback

## What This Notebook Shows:
1. **ROI Detection** - Visual verification that we found the right regions
2. **Baseline Method** - Polar correlation map (spacing × angle)
3. **SRS Map** - Where periodic structures are (Structural Repetition Score)
4. **Autocorrelation** - 1D profile with detected peaks
5. **FFT Power Spectrum** - Frequency analysis with peaks

## What is SRS?
**SRS = Structural Repetition Score**

For each location (x,y), SRS measures: *"Does the pattern at this location repeat nearby?"*
- Take a small block at (x,y)
- Search nearby for similar blocks
- SRS = maximum correlation found
- **High SRS = periodic structure** (pattern repeats)
- **Low SRS = non-periodic** (random/noise)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, ndimage, optimize
from scipy.signal import find_peaks
import tifffile
import skimage as ski
from skimage.transform import radon
from skimage.feature import match_template
import cv2
import warnings
warnings.filterwarnings('ignore')

print("Imports successful!")

## Configuration

In [ ]:
# ============================================================
# CONFIGURE YOUR IMAGE HERE
# ============================================================

IMAGE_PATH = "dImage3.tif"  # Change this to your image

# Pixel size in nm
PIXEL_SIZE_NM = 20.4

# Expected periodicity (Xu et al. 2013 - spectrin MPS)
EXPECTED_PERIOD_NM = 190
EXPECTED_PERIOD_PX = EXPECTED_PERIOD_NM / PIXEL_SIZE_NM  # ~9.3 pixels

# Search range for periodicity
MIN_PERIOD_NM = 100
MAX_PERIOD_NM = 350

# SReD Parameters (tuned for ~190 nm periodicity)
SRED_BLOCK_SIZE_PX = 10  # ~1 period = best for detecting periodicity
SRED_SEARCH_RADIUS_PX = 30  # ~3 periods
SRED_STRIDE_PX = 3  # Sampling density

# ============================================================

print(f"Configuration:")
print(f"  Image: {IMAGE_PATH}")
print(f"  Pixel size: {PIXEL_SIZE_NM} nm")
print(f"  Expected period: {EXPECTED_PERIOD_NM} nm ({EXPECTED_PERIOD_PX:.1f} px)")
print(f"  Search range: {MIN_PERIOD_NM} - {MAX_PERIOD_NM} nm")
print(f"  SReD block size: {SRED_BLOCK_SIZE_PX} px ({SRED_BLOCK_SIZE_PX * PIXEL_SIZE_NM:.0f} nm)")

---
## Load Image

In [ ]:
# Load image
image = tifffile.imread(IMAGE_PATH).astype(np.float64)
print(f"Image shape: {image.shape}")
print(f"Image size: {image.shape[0] * PIXEL_SIZE_NM / 1000:.2f} × {image.shape[1] * PIXEL_SIZE_NM / 1000:.2f} µm")

# Display
fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(image, cmap='gray')
ax.set_title(f'{IMAGE_PATH}\nSize: {image.shape}', fontsize=14)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

---
## Baseline Method: Kernel Matching Functions

In [ ]:
def compute_kernel(spacing, theta=0.0, sigma=2.0, nxy=32, w12=(9, 20)):
    """Create periodic stripe kernel."""
    w1, w2 = w12
    w2_scaled = w2 / max(spacing, 1)
    
    x, y = np.mgrid[-nxy//2:nxy//2:1, -nxy//2:nxy//2:1]
    pos = np.dstack((x, y))
    
    rv = stats.multivariate_normal(
        mean=[0.0, 0.0],
        cov=[[(nxy/w1)**2.0, 0.0], [0.0, (nxy/w2_scaled)**2.0]],
        allow_singular=False
    )
    w = rv.pdf(pos).astype(np.float64)
    
    stripes = np.zeros_like(w, dtype=np.float64)
    stripes[:, nxy//2 - spacing//2] = 1
    stripes[:, nxy//2 + spacing//2 + spacing % 2] = 1
    
    kernel0 = stripes * w
    kernelb = ndimage.gaussian_filter(kernel0, sigma)
    kernel = ndimage.rotate(kernelb, theta, reshape=False, order=0)
    kernel /= (kernel.sum() + 1e-10)
    
    return kernel


def get_correlation(img, spacings, angles, sigma=0.8, nxy=64, w12=(9, 20)):
    """Compute correlation with periodic kernels."""
    na, ns = len(angles), len(spacings)
    nx, ny = img.shape
    correlation_results = np.zeros((ns, na, nx, ny), dtype=np.float64)
    
    img_float = img.astype(np.float32)
    
    for id, spacing in enumerate(spacings):
        for ia, theta in enumerate(angles):
            kernel = compute_kernel(spacing=spacing, theta=theta, sigma=sigma, nxy=nxy, w12=w12)
            kernel = kernel.astype(np.float32)
            kernel /= (np.sum(kernel) + 1e-10)
            
            s = img.sum() + 1e-10
            kernel_scaled = s * kernel
            correlation_result = cv2.filter2D(src=img_float, kernel=kernel_scaled, ddepth=-1)
            correlation_results[id, ia, :, :] = correlation_result
    
    return correlation_results

print("Baseline functions defined")

---
## Step 1: ROI Detection (Coarse Kernel Matching)

Find regions with periodic structure using coarse template matching.

In [ ]:
# Coarse search to find ROIs
print("Running coarse ROI detection...")

angles_coarse = np.linspace(-90, 90, 12)
spacings_coarse = np.arange(6, 14, 2)  # 6-12 pixels ≈ 120-250 nm

print(f"  Testing spacings: {spacings_coarse} px ({spacings_coarse * PIXEL_SIZE_NM} nm)")
print(f"  Testing angles: {angles_coarse}°")

nxy = min(48, min(image.shape) // 4)
correlations_coarse = get_correlation(image.astype(np.float64), spacings_coarse, angles_coarse, 
                                       sigma=0.8, nxy=nxy)

# Sum over all spacings and angles
corr_sum = correlations_coarse.sum(axis=(0, 1))

# Threshold and find regions
threshold = ski.filters.threshold_otsu(corr_sum)
binary = corr_sum > threshold
binary = ski.segmentation.clear_border(binary)
labels = ski.measure.label(binary)

regions = [r for r in ski.measure.regionprops(labels) if r.area >= 1500]
print(f"\nFound {len(regions)} ROIs")

In [ ]:
# VISUALIZE: ROI Detection
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original image
ax = axes[0]
ax.imshow(image, cmap='gray')
ax.set_title('Original Image', fontsize=12, fontweight='bold')

# Correlation map
ax = axes[1]
im = ax.imshow(corr_sum, cmap='hot')
ax.set_title('Correlation Sum (all spacings/angles)\nBright = periodic structure', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

# ROIs
ax = axes[2]
ax.imshow(image, cmap='gray')
for idx, region in enumerate(regions):
    minr, minc, maxr, maxc = region.bbox
    rect = plt.Rectangle((minc, minr), maxc-minc, maxr-minr,
                          fill=False, edgecolor='lime', linewidth=2)
    ax.add_patch(rect)
    ax.text(minc, minr-5, f'ROI {idx+1}', color='lime', fontsize=10, fontweight='bold')
ax.set_title(f'Detected ROIs: {len(regions)}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('01_roi_detection.png', dpi=150)
plt.show()

print("\nROI Details:")
for idx, region in enumerate(regions):
    minr, minc, maxr, maxc = region.bbox
    print(f"  ROI {idx+1}: area={region.area}, bbox=({minr},{minc}) to ({maxr},{maxc})")

---
## Step 2: Analyze Each ROI with Full Visualization

In [ ]:
def analyze_roi_with_visualization(roi, roi_idx, pixel_size_nm, min_period_nm, max_period_nm):
    """
    Complete analysis of one ROI with full visualization.
    """
    print(f"\n{'='*70}")
    print(f"ROI {roi_idx}: Shape {roi.shape}")
    print(f"{'='*70}")
    
    # =========================================================
    # BASELINE METHOD: Kernel Matching
    # =========================================================
    print("\n1. BASELINE (Kernel Matching)...")
    
    min_spacing = max(5, int(min_period_nm / pixel_size_nm))
    max_spacing = min(18, int(max_period_nm / pixel_size_nm))
    
    spacings_px = np.arange(min_spacing, max_spacing + 1, 1)
    angles_deg = np.arange(0, 181, 5)
    spacings_nm = spacings_px * pixel_size_nm
    
    roi_norm = roi.astype(np.float64)
    roi_norm = roi_norm / (roi_norm.sum() + 1e-10)
    
    nxy = min(64, min(roi.shape) // 2)
    correlations = get_correlation(roi_norm, spacings_px, angles_deg, 
                                    sigma=0.8, nxy=nxy, w12=(9, 20))
    
    # Correlation map (spacing × angle)
    cm = correlations.sum(axis=(2, 3))
    cm_norm = (cm - cm.min()) / (cm.max() - cm.min() + 1e-10)
    
    # Find best angle
    angle_profile = cm.sum(axis=0)
    optimal_angle_idx = np.argmax(angle_profile)
    optimal_angle = angles_deg[optimal_angle_idx]
    
    # Spacing profile at best angle
    spacing_profile = cm[:, optimal_angle_idx]
    spacing_profile_norm = (spacing_profile - spacing_profile.min()) / (spacing_profile.max() - spacing_profile.min() + 1e-10)
    
    # Find peak
    peak_idx = np.argmax(spacing_profile_norm)
    baseline_period = spacings_nm[peak_idx]
    baseline_quality = spacing_profile_norm[peak_idx]
    
    print(f"   Period: {baseline_period:.1f} nm")
    print(f"   Angle: {optimal_angle}°")
    print(f"   Quality: {baseline_quality:.3f}")
    
    # =========================================================
    # SReD METHOD: Block Matching (SRS Map)
    # =========================================================
    print("\n2. SReD-style (SRS Map)...")
    
    roi_sred = (roi - roi.mean()) / (roi.std() + 1e-10)
    
    block_size = SRED_BLOCK_SIZE_PX
    search_radius = SRED_SEARCH_RADIUS_PX
    stride = SRED_STRIDE_PX
    half_block = block_size // 2
    
    srs_map = np.zeros_like(roi_sred)
    
    for y in range(half_block + search_radius, roi.shape[0] - half_block - search_radius, stride):
        for x in range(half_block + search_radius, roi.shape[1] - half_block - search_radius, stride):
            # Reference block
            ref = roi_sred[y-half_block:y+half_block, x-half_block:x+half_block]
            if ref.size == 0 or ref.shape[0] < block_size or ref.shape[1] < block_size:
                continue
            
            # Search region
            y1, y2 = max(0, y - search_radius), min(roi.shape[0], y + search_radius)
            x1, x2 = max(0, x - search_radius), min(roi.shape[1], x + search_radius)
            search = roi_sred[y1:y2, x1:x2]
            
            if search.shape[0] > ref.shape[0] and search.shape[1] > ref.shape[1]:
                try:
                    result = match_template(search, ref, pad_input=True)
                    # Mask center (self-match)
                    cy, cx = result.shape[0]//2, result.shape[1]//2
                    mask_r = half_block + 2
                    result[max(0,cy-mask_r):min(result.shape[0],cy+mask_r+1), 
                           max(0,cx-mask_r):min(result.shape[1],cx+mask_r+1)] = 0
                    srs_map[y, x] = np.max(result)
                except:
                    pass
    
    # SRS statistics
    valid_srs = srs_map[srs_map > 0]
    if len(valid_srs) > 0:
        srs_mean = np.mean(valid_srs)
        srs_max = np.max(valid_srs)
    else:
        srs_mean = 0
        srs_max = 0
    
    print(f"   SRS mean: {srs_mean:.3f}")
    print(f"   SRS max: {srs_max:.3f}")
    
    # =========================================================
    # AUTOCORRELATION
    # =========================================================
    print("\n3. AUTOCORRELATION...")
    
    # Get best 1D profile using Radon
    roi_radon = (roi - roi.mean()) / (roi.std() + 1e-10)
    theta = np.linspace(0, 180, 90, endpoint=False)
    sinogram = radon(roi_radon, theta=theta, circle=True)
    variances = np.var(sinogram, axis=0)
    best_angle_idx = np.argmax(variances)
    autocorr_angle = theta[best_angle_idx]
    profile = sinogram[:, best_angle_idx]
    profile = profile - profile.mean()
    
    # Autocorrelation
    autocorr = np.correlate(profile, profile, mode='full')
    autocorr = autocorr[len(autocorr)//2:]
    autocorr = autocorr / (autocorr[0] + 1e-10)
    lags_nm = np.arange(len(autocorr)) * pixel_size_nm
    
    # Find peak in range
    min_lag = int(min_period_nm / pixel_size_nm)
    max_lag = min(len(autocorr)-1, int(max_period_nm / pixel_size_nm))
    
    if max_lag > min_lag:
        segment = autocorr[min_lag:max_lag]
        peaks, props = find_peaks(segment, height=0, distance=3)
        
        if len(peaks) > 0:
            best_peak_idx = np.argmax(props['peak_heights'])
            autocorr_period = (peaks[best_peak_idx] + min_lag) * pixel_size_nm
            autocorr_quality = props['peak_heights'][best_peak_idx]
        else:
            peak_idx = np.argmax(segment)
            autocorr_period = (peak_idx + min_lag) * pixel_size_nm
            autocorr_quality = segment[peak_idx]
    else:
        autocorr_period = 190
        autocorr_quality = 0
        peaks = []
    
    print(f"   Period: {autocorr_period:.1f} nm")
    print(f"   Angle: {autocorr_angle:.1f}°")
    print(f"   Quality: {autocorr_quality:.3f}")
    
    # =========================================================
    # FFT
    # =========================================================
    print("\n4. FFT...")
    
    n = len(profile)
    fft = np.fft.fft(profile)
    power = np.abs(fft[:n//2])**2
    freqs = np.fft.fftfreq(n, d=pixel_size_nm)[:n//2]
    periods_fft = np.where(freqs > 0, 1 / freqs, np.inf)
    
    # Find peak in valid range
    valid_mask = (periods_fft >= min_period_nm) & (periods_fft <= max_period_nm)
    valid_indices = np.where(valid_mask)[0]
    
    if len(valid_indices) > 0:
        valid_power = power[valid_indices]
        best_idx = valid_indices[np.argmax(valid_power)]
        fft_period = periods_fft[best_idx]
        fft_quality = (power[best_idx] - np.median(power[valid_mask])) / (power[best_idx] + 1e-10)
    else:
        fft_period = 190
        fft_quality = 0
    
    print(f"   Period: {fft_period:.1f} nm")
    print(f"   Quality: {fft_quality:.3f}")
    
    # =========================================================
    # VISUALIZATION
    # =========================================================
    fig = plt.figure(figsize=(20, 16))
    
    # Row 1: ROI and SRS Map
    ax1 = plt.subplot(3, 4, 1)
    ax1.imshow(roi, cmap='gray')
    ax1.set_title(f'ROI {roi_idx}\n{roi.shape}', fontsize=11, fontweight='bold')
    ax1.axis('off')
    
    ax2 = plt.subplot(3, 4, 2)
    im = ax2.imshow(srs_map, cmap='hot', vmin=0, vmax=1)
    ax2.set_title(f'SRS Map\nMean={srs_mean:.3f}, Max={srs_max:.3f}', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax2, shrink=0.8)
    ax2.axis('off')
    
    # SRS histogram
    ax3 = plt.subplot(3, 4, 3)
    if len(valid_srs) > 0:
        ax3.hist(valid_srs, bins=30, color='coral', edgecolor='black', alpha=0.7)
        ax3.axvline(srs_mean, color='red', linestyle='--', linewidth=2, label=f'Mean: {srs_mean:.3f}')
    ax3.set_xlabel('SRS Value')
    ax3.set_ylabel('Count')
    ax3.set_title('SRS Distribution', fontsize=11, fontweight='bold')
    ax3.legend()
    
    # Overlay: ROI with high-SRS regions
    ax4 = plt.subplot(3, 4, 4)
    ax4.imshow(roi, cmap='gray')
    if srs_max > 0:
        high_srs = srs_map > 0.5 * srs_max
        ax4.contour(high_srs, colors='lime', linewidths=1)
    ax4.set_title('ROI with High-SRS Regions', fontsize=11, fontweight='bold')
    ax4.axis('off')
    
    # Row 2: Baseline Method
    # Correlation heatmap
    ax5 = plt.subplot(3, 4, 5)
    im = ax5.imshow(cm_norm, aspect='auto', origin='lower',
                    extent=[angles_deg[0], angles_deg[-1], spacings_nm[0], spacings_nm[-1]],
                    cmap='viridis')
    ax5.axhline(EXPECTED_PERIOD_NM, color='red', linestyle='--', linewidth=1.5, label=f'Expected: {EXPECTED_PERIOD_NM} nm')
    ax5.axhline(baseline_period, color='lime', linestyle=':', linewidth=2, label=f'Detected: {baseline_period:.0f} nm')
    ax5.set_xlabel('Angle (°)')
    ax5.set_ylabel('Spacing (nm)')
    ax5.set_title('Baseline: Correlation Map', fontsize=11, fontweight='bold')
    ax5.legend(loc='upper right', fontsize=8)
    plt.colorbar(im, ax=ax5, shrink=0.8)
    
    # Polar plot
    ax6 = plt.subplot(3, 4, 6, projection='polar')
    r, theta_mesh = np.meshgrid(spacings_nm, np.radians(angles_deg))
    c = ax6.contourf(theta_mesh, r, cm_norm.T, levels=20, cmap='viridis')
    ax6.set_title('Baseline: Polar View', fontsize=11, fontweight='bold')
    
    # Spacing profile
    ax7 = plt.subplot(3, 4, 7)
    ax7.plot(spacings_nm, spacing_profile_norm, 'b-o', linewidth=2, markersize=5)
    ax7.axvline(EXPECTED_PERIOD_NM, color='red', linestyle='--', linewidth=1.5, label=f'Expected: {EXPECTED_PERIOD_NM} nm')
    ax7.axvline(baseline_period, color='lime', linestyle=':', linewidth=2, label=f'Detected: {baseline_period:.0f} nm')
    ax7.fill_between(spacings_nm, 0, spacing_profile_norm, alpha=0.3)
    ax7.set_xlabel('Spacing (nm)')
    ax7.set_ylabel('Correlation (norm)')
    ax7.set_title(f'Baseline: Spacing Profile @ {optimal_angle}°', fontsize=11, fontweight='bold')
    ax7.legend(fontsize=8)
    ax7.grid(True, alpha=0.3)
    
    # Angle profile
    ax8 = plt.subplot(3, 4, 8)
    angle_profile_norm = (angle_profile - angle_profile.min()) / (angle_profile.max() - angle_profile.min() + 1e-10)
    ax8.plot(angles_deg, angle_profile_norm, 'g-o', linewidth=2, markersize=4)
    ax8.axvline(optimal_angle, color='lime', linestyle=':', linewidth=2, label=f'Best: {optimal_angle}°')
    ax8.set_xlabel('Angle (°)')
    ax8.set_ylabel('Correlation (norm)')
    ax8.set_title('Baseline: Angle Profile', fontsize=11, fontweight='bold')
    ax8.legend(fontsize=8)
    ax8.grid(True, alpha=0.3)
    
    # Row 3: Autocorrelation and FFT
    # 1D Profile
    ax9 = plt.subplot(3, 4, 9)
    x_profile = np.arange(len(profile)) * pixel_size_nm
    ax9.plot(x_profile, profile, 'b-', linewidth=1)
    ax9.set_xlabel('Position (nm)')
    ax9.set_ylabel('Intensity')
    ax9.set_title(f'1D Profile @ {autocorr_angle:.0f}°', fontsize=11, fontweight='bold')
    ax9.grid(True, alpha=0.3)
    
    # Autocorrelation
    ax10 = plt.subplot(3, 4, 10)
    ax10.plot(lags_nm[:max_lag+20], autocorr[:max_lag+20], 'b-', linewidth=1.5)
    ax10.axvline(EXPECTED_PERIOD_NM, color='red', linestyle='--', linewidth=1.5, label=f'Expected: {EXPECTED_PERIOD_NM} nm')
    ax10.axvline(autocorr_period, color='lime', linestyle=':', linewidth=2, label=f'Detected: {autocorr_period:.0f} nm')
    ax10.axhline(0, color='gray', linestyle='-', linewidth=0.5)
    ax10.axvspan(min_period_nm, max_period_nm, alpha=0.1, color='blue', label='Search range')
    ax10.set_xlabel('Lag (nm)')
    ax10.set_ylabel('Autocorrelation')
    ax10.set_title(f'Autocorr: Period={autocorr_period:.0f} nm', fontsize=11, fontweight='bold')
    ax10.legend(fontsize=8)
    ax10.grid(True, alpha=0.3)
    
    # FFT Power Spectrum
    ax11 = plt.subplot(3, 4, 11)
    # Plot vs period (not frequency)
    valid_plot = (periods_fft > 50) & (periods_fft < 500)
    ax11.plot(periods_fft[valid_plot], power[valid_plot], 'b-', linewidth=1.5)
    ax11.axvline(EXPECTED_PERIOD_NM, color='red', linestyle='--', linewidth=1.5, label=f'Expected: {EXPECTED_PERIOD_NM} nm')
    ax11.axvline(fft_period, color='lime', linestyle=':', linewidth=2, label=f'Detected: {fft_period:.0f} nm')
    ax11.axvspan(min_period_nm, max_period_nm, alpha=0.1, color='blue', label='Search range')
    ax11.set_xlabel('Period (nm)')
    ax11.set_ylabel('Power')
    ax11.set_title(f'FFT: Period={fft_period:.0f} nm', fontsize=11, fontweight='bold')
    ax11.legend(fontsize=8)
    ax11.grid(True, alpha=0.3)
    ax11.set_xlim(50, 500)
    
    # Summary
    ax12 = plt.subplot(3, 4, 12)
    ax12.axis('off')
    summary_text = f"""
    SUMMARY - ROI {roi_idx}
    {'='*30}
    
    Expected Period: {EXPECTED_PERIOD_NM} nm
    
    BASELINE (Kernel):
      Period: {baseline_period:.1f} nm
      Angle: {optimal_angle}°
      Quality: {baseline_quality:.3f}
    
    AUTOCORRELATION:
      Period: {autocorr_period:.1f} nm
      Angle: {autocorr_angle:.0f}°
      Quality: {autocorr_quality:.3f}
    
    FFT:
      Period: {fft_period:.1f} nm
      Quality: {fft_quality:.3f}
    
    SReD (SRS):
      Mean SRS: {srs_mean:.3f}
      Max SRS: {srs_max:.3f}
    """
    ax12.text(0.1, 0.9, summary_text, transform=ax12.transAxes, fontsize=11,
              verticalalignment='top', fontfamily='monospace',
              bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    plt.suptitle(f'Complete Analysis: ROI {roi_idx}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'02_roi_{roi_idx}_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return {
        'roi_idx': roi_idx,
        'baseline_period': baseline_period,
        'baseline_angle': optimal_angle,
        'baseline_quality': baseline_quality,
        'autocorr_period': autocorr_period,
        'autocorr_angle': autocorr_angle,
        'autocorr_quality': autocorr_quality,
        'fft_period': fft_period,
        'fft_quality': fft_quality,
        'srs_mean': srs_mean,
        'srs_max': srs_max,
        'srs_map': srs_map,
        'correlation_map': cm_norm
    }

print("Analysis function defined")

---
## Run Analysis on All ROIs

In [ ]:
# Analyze each ROI
all_results = []

for idx, region in enumerate(regions):
    # Extract ROI
    minr, minc, maxr, maxc = region.bbox
    padding = 10
    roi = image[
        max(0, minr-padding):min(image.shape[0], maxr+padding),
        max(0, minc-padding):min(image.shape[1], maxc+padding)
    ].copy()
    
    if roi.size < 500 or min(roi.shape) < 30:
        print(f"ROI {idx+1} too small, skipping")
        continue
    
    # Analyze
    result = analyze_roi_with_visualization(
        roi, idx+1, PIXEL_SIZE_NM, MIN_PERIOD_NM, MAX_PERIOD_NM
    )
    all_results.append(result)

---
## Summary Table

In [ ]:
# Create summary table
if all_results:
    print("\n" + "="*100)
    print("SUMMARY TABLE")
    print("="*100)
    print(f"\nExpected Period: {EXPECTED_PERIOD_NM} nm (Xu et al. 2013 - Spectrin MPS)\n")
    
    print(f"{'ROI':<5} | {'Baseline':<12} | {'Autocorr':<12} | {'FFT':<12} | {'SRS Mean':<10} | {'Agreement'}")
    print(f"{'':<5} | {'Period (nm)':<12} | {'Period (nm)':<12} | {'Period (nm)':<12} | {'(0-1)':<10} | {'with 190nm'}")
    print("-"*85)
    
    for r in all_results:
        # Check if any method detected ~190 nm (within ±30 nm)
        periods = [r['baseline_period'], r['autocorr_period'], r['fft_period']]
        agreements = sum(1 for p in periods if abs(p - EXPECTED_PERIOD_NM) < 30)
        agreement_str = f"{agreements}/3 methods"
        
        print(f"{r['roi_idx']:<5} | {r['baseline_period']:>10.1f}nm | {r['autocorr_period']:>10.1f}nm | {r['fft_period']:>10.1f}nm | {r['srs_mean']:>10.3f} | {agreement_str}")
    
    print("\n" + "="*100)
    print("INTERPRETATION:")
    print("="*100)
    print("""
    - Period should be ~190 nm for nodes of Ranvier (spectrin MPS)
    - If detected period is very different, ROI may not be a real node
    - SRS > 0.5 indicates strong periodic structure
    - Quality > 0.3 indicates reliable detection
    - Look at the polar plot to see if periodicity is real (bright spot at ~190 nm radius)
    """)

---
## Visual: Compare All ROIs Side by Side

In [ ]:
if len(all_results) > 1:
    n_rois = len(all_results)
    fig, axes = plt.subplots(2, n_rois, figsize=(5*n_rois, 10))
    if n_rois == 1:
        axes = axes.reshape(2, 1)
    
    for i, r in enumerate(all_results):
        # ROI image
        region = regions[r['roi_idx']-1]
        minr, minc, maxr, maxc = region.bbox
        padding = 10
        roi = image[
            max(0, minr-padding):min(image.shape[0], maxr+padding),
            max(0, minc-padding):min(image.shape[1], maxc+padding)
        ].copy()
        
        axes[0, i].imshow(roi, cmap='gray')
        axes[0, i].set_title(f"ROI {r['roi_idx']}\nBaseline: {r['baseline_period']:.0f} nm", fontsize=10)
        axes[0, i].axis('off')
        
        # SRS map
        axes[1, i].imshow(r['srs_map'], cmap='hot', vmin=0, vmax=1)
        axes[1, i].set_title(f"SRS Map\nMean: {r['srs_mean']:.3f}", fontsize=10)
        axes[1, i].axis('off')
    
    plt.suptitle('All ROIs Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('03_all_rois_comparison.png', dpi=150)
    plt.show()

---
## What is SRS? (Detailed Explanation)

```
SRS = Structural Repetition Score

For each location (x,y):

1. Extract a BLOCK around (x,y)
   ┌─────────┐
   │ ▒▒▒▒▒▒▒ │  ← Reference block (~10×10 pixels)
   │ ▒▒▒▒▒▒▒ │
   │ ▒▒▒▒▒▒▒ │
   └─────────┘

2. SEARCH nearby for similar blocks
   ┌───────────────────────────────┐
   │                               │
   │    ?    ?    ?    ?    ?      │  ← Search region (~60×60 pixels)
   │    ?    ┌───┐    ?    ?      │
   │    ?    │REF│    ?    ?      │  ← Reference at center
   │    ?    └───┘    ?    ?      │
   │    ?    ?    ?    ?    ?      │
   │                               │
   └───────────────────────────────┘

3. Compute CORRELATION with each location
   - High correlation = pattern repeats
   - SRS = MAXIMUM correlation found (excluding self)

4. Result: SRS MAP
   - HIGH SRS (bright) = periodic structure
   - LOW SRS (dark) = random/noise

Parameters (for 190 nm periodicity at 20.4 nm/pixel):
   - Block size = ~10 pixels (1 period)
   - Search radius = ~30 pixels (3 periods)
   - Stride = 3 pixels (sampling density)
```